# Momentum in Systems: Interactive Collision Sandbox

**Purpose:** Use an idealized one-dimensional collision model to investigate how the momenta of individual objects can change while the total momentum of an isolated system remains constant.

## Learning objective

Calculate and compare the total momentum of a two-object system before and after elastic and inelastic collisions, then explain the result using system boundaries, direction, and momentum transfer.

## Digital-skills objective

Manipulate a computational model, compare predictions with generated evidence, inspect multiple representations, and identify assumptions or limitations built into the model.

## Sign convention and model assumptions

- Rightward velocity is positive; leftward velocity is negative.
- Cart 1 begins to the left of Cart 2.
- A collision occurs only when Cart 1 is catching Cart 2, so \(u_1 > u_2\).
- The system is isolated during the collision.
- Motion is one-dimensional.
- Masses remain constant.
- The coefficient of restitution \(e\) ranges from 0 to 1:
  - \(e=1\): elastic collision
  - \(e=0\): perfectly inelastic collision
  - \(0<e<1\): partially inelastic collision

**Workflow:** Make a prediction first. Then select **Run collision and reveal results**.

In [10]:
from dataclasses import dataclass
import math

import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

In [12]:
@dataclass(frozen=True)
class CollisionResult:
    m1: float
    u1: float
    m2: float
    u2: float
    e: float
    v1: float
    v2: float
    p1_i: float
    p2_i: float
    p_total_i: float
    p1_f: float
    p2_f: float
    p_total_f: float
    ke_i: float
    ke_f: float


def collision_outcome(
    m1: float,
    u1: float,
    m2: float,
    u2: float,
    e: float,
) -> CollisionResult:
    """Return the outcome of an idealized 1-D collision.

    The calculation uses conservation of momentum together with
    the coefficient-of-restitution relation:

        e = (v2 - v1) / (u1 - u2)
    """
    if not all(math.isfinite(x) for x in (m1, u1, m2, u2, e)):
        raise ValueError("All inputs must be finite numbers.")
    if m1 <= 0 or m2 <= 0:
        raise ValueError("Both masses must be greater than zero.")
    if not 0 <= e <= 1:
        raise ValueError("The coefficient of restitution must be between 0 and 1.")

    total_mass = m1 + m2
    relative_approach_speed = u1 - u2

    v1 = (
        m1 * u1
        + m2 * u2
        - m2 * e * relative_approach_speed
    ) / total_mass

    v2 = (
        m1 * u1
        + m2 * u2
        + m1 * e * relative_approach_speed
    ) / total_mass

    p1_i = m1 * u1
    p2_i = m2 * u2
    p1_f = m1 * v1
    p2_f = m2 * v2

    ke_i = 0.5 * m1 * u1**2 + 0.5 * m2 * u2**2
    ke_f = 0.5 * m1 * v1**2 + 0.5 * m2 * v2**2

    return CollisionResult(
        m1=m1,
        u1=u1,
        m2=m2,
        u2=u2,
        e=e,
        v1=v1,
        v2=v2,
        p1_i=p1_i,
        p2_i=p2_i,
        p_total_i=p1_i + p2_i,
        p1_f=p1_f,
        p2_f=p2_f,
        p_total_f=p1_f + p2_f,
        ke_i=ke_i,
        ke_f=ke_f,
    )


def direction_label(value: float, tolerance: float = 1e-9) -> str:
    if value > tolerance:
        return "right"
    if value < -tolerance:
        return "left"
    return "at rest"


def signed(value: float, digits: int = 3) -> str:
    return f"{value:+.{digits}f}"

In [13]:
label_style = {"description_width": "145px"}
full_width = widgets.Layout(width="100%")
half_width = widgets.Layout(width="48%", min_width="280px")
slider_layout = widgets.Layout(width="100%")

mass_1 = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description="Cart 1 mass (kg)",
    continuous_update=False,
    style=label_style,
    layout=slider_layout,
)

velocity_1 = widgets.FloatSlider(
    value=2.0,
    min=-5.0,
    max=5.0,
    step=0.1,
    description="Cart 1 initial v",
    continuous_update=False,
    readout_format=".1f",
    style=label_style,
    layout=slider_layout,
)

mass_2 = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description="Cart 2 mass (kg)",
    continuous_update=False,
    style=label_style,
    layout=slider_layout,
)

velocity_2 = widgets.FloatSlider(
    value=0.0,
    min=-5.0,
    max=5.0,
    step=0.1,
    description="Cart 2 initial v",
    continuous_update=False,
    readout_format=".1f",
    style=label_style,
    layout=slider_layout,
)

collision_type = widgets.Dropdown(
    options=[
        ("Elastic (e = 1)", "elastic"),
        ("Perfectly inelastic (e = 0)", "inelastic"),
        ("Partially inelastic (choose e)", "custom"),
    ],
    value="elastic",
    description="Collision type",
    style=label_style,
    layout=full_width,
)

restitution = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Restitution e",
    continuous_update=False,
    disabled=True,
    style=label_style,
    layout=slider_layout,
)

prediction_v1 = widgets.FloatText(
    value=0.0,
    description="Predicted Cart 1 final velocity",
    style=label_style,
    layout=full_width,
)

prediction_v2 = widgets.FloatText(
    value=0.0,
    description="Predicted Cart 2 final velocity",
    style=label_style,
    layout=full_width,
)

prediction_reasoning = widgets.Textarea(
    value="",
    placeholder="Explain your prediction using mass, velocity, direction, and system momentum.",
    description="Reasoning",
    style=label_style,
    layout=widgets.Layout(width="100%", height="95px"),
)

run_button = widgets.Button(
    description="Run collision and reveal results",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="300px", height="44px"),
)

reset_button = widgets.Button(
    description="Reset example",
    icon="refresh",
    layout=widgets.Layout(width="160px", height="44px"),
)

status_output = widgets.HTML()
summary_output = widgets.HTML()
table_output = widgets.HTML()
prediction_output = widgets.HTML()
plot_output = widgets.Output()

In [16]:
def update_restitution(change=None):
    selected = collision_type.value
    if selected == "elastic":
        restitution.value = 1.0
        restitution.disabled = True
    elif selected == "inelastic":
        restitution.value = 0.0
        restitution.disabled = True
    else:
        restitution.disabled = False


def make_results_table(result: CollisionResult) -> str:
    data = pd.DataFrame(
        {
            "Quantity": [
                "Cart 1 velocity (m/s)",
                "Cart 2 velocity (m/s)",
                "Cart 1 momentum (kg·m/s)",
                "Cart 2 momentum (kg·m/s)",
                "System momentum (kg·m/s)",
                "System kinetic energy (J)",
            ],
            "Before": [
                result.u1,
                result.u2,
                result.p1_i,
                result.p2_i,
                result.p_total_i,
                result.ke_i,
            ],
            "After": [
                result.v1,
                result.v2,
                result.p1_f,
                result.p2_f,
                result.p_total_f,
                result.ke_f,
            ],
        }
    )
    return data.to_html(
        index=False,
        float_format=lambda value: f"{value:.3f}",
        border=0,
        classes="momentum-table",
    )


def make_plots(result: CollisionResult) -> None:
    with plot_output:
        clear_output(wait=True)

        momentum_labels = ["Cart 1", "Cart 2", "System total"]
        before_momentum = [
            result.p1_i,
            result.p2_i,
            result.p_total_i,
        ]
        after_momentum = [
            result.p1_f,
            result.p2_f,
            result.p_total_f,
        ]

        x = range(len(momentum_labels))
        width = 0.36

        fig, ax = plt.subplots(figsize=(8, 4.4))
        before_positions = [position - width / 2 for position in x]
        after_positions = [position + width / 2 for position in x]

        before_bars = ax.bar(
            before_positions,
            before_momentum,
            width,
            label="Before collision",
        )
        after_bars = ax.bar(
            after_positions,
            after_momentum,
            width,
            label="After collision",
        )

        ax.axhline(0, linewidth=1)
        ax.set_xticks(list(x), momentum_labels)
        ax.set_ylabel("Momentum (kg·m/s)")
        ax.set_title("Momentum of each cart and of the system")
        ax.legend()
        ax.bar_label(before_bars, fmt="%.2f", padding=3)
        ax.bar_label(after_bars, fmt="%.2f", padding=3)
        fig.tight_layout()
        plt.show()
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(6.5, 3.8))
        energy_bars = ax.bar(
            ["Before collision", "After collision"],
            [result.ke_i, result.ke_f],
        )
        ax.set_ylabel("Kinetic energy (J)")
        ax.set_title("System kinetic energy")
        ax.bar_label(energy_bars, fmt="%.3f", padding=3)
        fig.tight_layout()
        plt.show()
        plt.close(fig)


def run_collision(_button=None):
    status_output.value = ""
    summary_output.value = ""
    table_output.value = ""
    prediction_output.value = ""
    with plot_output:
        clear_output(wait=True)

    if velocity_1.value <= velocity_2.value:
        status_output.value = (
            "<div style='padding:12px;border-left:5px solid #b45309;"
            "background:#F7E065;'>"
            "<strong>Adjust the initial velocities.</strong> "
            "Cart 1 begins to the left of Cart 2, so Cart 1 must have "
            "a greater initial velocity than Cart 2 in order to catch it "
            "and collide: <code>u₁ &gt; u₂</code>."
            "</div>"
        )
        return

    try:
        result = collision_outcome(
            m1=mass_1.value,
            u1=velocity_1.value,
            m2=mass_2.value,
            u2=velocity_2.value,
            e=restitution.value,
        )
    except ValueError as exc:
        status_output.value = (
            "<div style='padding:12px;border-left:5px solid #b91c1c;"
            "background:#F7E065;'>"
            f"<strong>Input error:</strong> {exc}"
            "</div>"
        )
        return

    momentum_difference = result.p_total_f - result.p_total_i
    energy_change = result.ke_f - result.ke_i
    energy_percent = (
        100 * energy_change / result.ke_i
        if result.ke_i > 1e-12
        else 0.0
    )

    collision_name = {
        "elastic": "elastic",
        "inelastic": "perfectly inelastic",
        "custom": "partially inelastic",
    }[collision_type.value]

    status_output.value = (
        "<div style='padding:12px;border-left:5px solid #15803d;"
        "background:#F7E065;'>"
        f"<strong>{collision_name.capitalize()} collision calculated.</strong> "
        "Now compare the individual quantities with the system totals."
        "</div>"
    )

    summary_output.value = f"""
    <div style="display:flex;flex-wrap:wrap;gap:12px;margin:12px 0;">
      <div style="flex:1;min-width:220px;padding:12px;border:1px solid #d1d5db;border-radius:8px;">
        <strong>Cart 1 after collision</strong><br>
        v₁f = {signed(result.v1)} m/s ({direction_label(result.v1)})<br>
        p₁f = {signed(result.p1_f)} kg·m/s
      </div>
      <div style="flex:1;min-width:220px;padding:12px;border:1px solid #d1d5db;border-radius:8px;">
        <strong>Cart 2 after collision</strong><br>
        v₂f = {signed(result.v2)} m/s ({direction_label(result.v2)})<br>
        p₂f = {signed(result.p2_f)} kg·m/s
      </div>
      <div style="flex:1;min-width:220px;padding:12px;border:1px solid #d1d5db;border-radius:8px;">
        <strong>System comparison</strong><br>
        Δp<sub>system</sub> = {momentum_difference:+.6f} kg·m/s<br>
        ΔKE = {energy_change:+.3f} J ({energy_percent:+.1f}%)
      </div>
    </div>
    """

    table_output.value = (
        "<h3>Before-and-after evidence</h3>"
        + make_results_table(result)
    )

    prediction_error_v1 = prediction_v1.value - result.v1
    prediction_error_v2 = prediction_v2.value - result.v2
    reasoning = prediction_reasoning.value.strip()

    prediction_output.value = f"""
    <div style="padding:12px;border:1px solid #d1d5db;border-radius:8px;margin-top:12px;">
      <h3 style="margin-top:0;">Prediction check</h3>
      <p>
        Cart 1 prediction error:
        <strong>{prediction_error_v1:+.3f} m/s</strong><br>
        Cart 2 prediction error:
        <strong>{prediction_error_v2:+.3f} m/s</strong>
      </p>
      <p><strong>Your reasoning:</strong><br>
        {reasoning if reasoning else "<em>No written reasoning was entered.</em>"}
      </p>
      <p>
        <strong>Interpretive question:</strong>
        How can the individual cart momenta change while the system
        momentum remains constant?
      </p>
    </div>
    """

    make_plots(result)


def reset_example(_button=None):
    mass_1.value = 1.0
    velocity_1.value = 2.0
    mass_2.value = 1.0
    velocity_2.value = 0.0
    collision_type.value = "elastic"
    restitution.value = 1.0
    prediction_v1.value = 0.0
    prediction_v2.value = 0.0
    prediction_reasoning.value = ""
    status_output.value = ""
    summary_output.value = ""
    table_output.value = ""
    prediction_output.value = ""
    with plot_output:
        clear_output(wait=True)


collision_type.observe(update_restitution, names="value")
run_button.on_click(run_collision)
reset_button.on_click(reset_example)
update_restitution()

In [21]:
title = widgets.HTML(
    value="""
    <div style="padding:16px 0 8px 0;">
      <h1 style="margin-bottom:4px;">Momentum in Systems</h1>
      <p style="font-size:1.05rem;margin-top:0;">
        Predict, model, and interpret a one-dimensional collision.
      </p>
    </div>
    """
)

directions = widgets.HTML(
    value="""
    <div style="padding:12px;border-left:5px solid #2563eb;background:#F7E065;margin-bottom:14px;">
      <strong>Before revealing the model output:</strong>
      choose the system parameters, predict both final velocities,
      and explain your reasoning. Rightward is positive.
    </div>
    """
)

cart_1_panel = widgets.VBox(
    [
        widgets.HTML("<h3>Cart 1: starts on the left</h3>"),
        mass_1,
        velocity_1,
    ],
    layout=half_width,
)

cart_2_panel = widgets.VBox(
    [
        widgets.HTML("<h3>Cart 2: starts on the right</h3>"),
        mass_2,
        velocity_2,
    ],
    layout=half_width,
)

carts_row = widgets.HBox(
    [cart_1_panel, cart_2_panel],
    layout=widgets.Layout(
        width="100%",
        display="flex",
        flex_flow="row wrap",
        justify_content="space-between",
        align_items="flex-start",
    ),
)

collision_panel = widgets.VBox(
    [
        widgets.HTML("<h3>Collision model</h3>"),
        collision_type,
        restitution,
    ],
    layout=widgets.Layout(width="100%"),
)

prediction_panel = widgets.VBox(
    [
        widgets.HTML("<h3>Your prediction</h3>"),
        prediction_v1,
        prediction_v2,
        prediction_reasoning,
    ],
    layout=widgets.Layout(width="100%"),
)

button_row = widgets.HBox(
    [run_button, reset_button],
    layout=widgets.Layout(
        width="100%",
        display="flex",
        flex_flow="row wrap",
        margin="10px 0 14px 0",
    ),
)

assumptions = widgets.HTML(
    value="""
    <div style="padding:12px;border:1px solid #d1d5db;border-radius:8px;margin-top:14px;">
      <h3 style="margin-top:0;">Inspect the Model Assumptions</h3>

      <ul>
        <li>The system contains both carts.</li>
        <li>External impulse during the collision is negligible.</li>
        <li>Motion is one-dimensional.</li>
        <li>Masses remain constant.</li>
        <li>The selected coefficient of restitution describes the collision.</li>
        <li>The equations do not model friction before or after the collision.</li>
      </ul>

      <p style="margin-bottom:0;">
        <strong>Critical question:</strong>
        Which of these assumptions would be least realistic in a physical cart experiment?
        Explain how violating that assumption could affect the model's results.
      </p>
    </div>
    """
)

student_challenges = widgets.HTML(
    value="""
    <div style="padding:12px;border:1px solid #d1d5db;border-radius:8px;margin-top:14px;">
      <h3 style="margin-top:0;">Suggested challenges</h3>
      <ol>
        <li>Create an elastic collision in which Cart 1 stops.</li>
        <li>Create a perfectly inelastic collision in which the joined carts move left.</li>
        <li>Create a collision with zero total initial momentum.</li>
        <li>Explain why momentum can be conserved when kinetic energy decreases.</li>
      </ol>
    </div>
    """
)

app = widgets.VBox(
    [
        title,
        directions,
        carts_row,
        collision_panel,
        prediction_panel,
        button_row,
        status_output,
        summary_output,
        table_output,
        plot_output,
        prediction_output,
        assumptions,
        student_challenges,
    ],
    layout=widgets.Layout(width="100%", max_width="1000px"),
)

display(app)